
# <font color="green">Observe Memory Management</font>

- Witness memory management in action.
- To this end, we are going to experiment with a program that repeatedly allocates a block of memory (an $S$-element array of 64-bit integers) and maintains a reference to the last $M$ allocated blocks via an outer array (an array of arrays). Perform $N > M$ allocations in total.
- Programs are given in C/C++ and in your assigned languages (Go, Julia, OCaml, Rust); measure the memory consumption of each program.
- Note: C/C++ version does not use free memory at all; observe what happens and compare it with other languages.
- Note: Julia has a significantly larger baseline memory footprint than the other languages due to its JIT compiler.

## Work

- For given $S$, $M$, and $N$, how much memory is reachable from the root once the outer array is fully populated?
- Measure actual memory consumption by running each program. Use the `/usr/bin/time` command and record the `maxresident` value. For example, the following output indicates that `ls` consumed 3996 kB:

```
$ /usr/bin/time ls
   ...
0.00user 0.00system 0:00.00elapsed 100%CPU (0avgtext+0avgdata 3996maxresident)k
0inputs+0outputs (0major+97minor)pagefaults 0swaps
```

- Suggested parameters: $S = 20000$ and $M = 1000$; vary $N$ from a small value up to 10000.
- Plot the results using online Excel, with $N$ on the $x$-axis and maximum resident set size on the $y$-axis.
- You are encouraged to automatically extract necessary values from the execution log.
- Compare results across languages by collaborating with your team members to put them in one Excel sheet.




# 1. AI tutor

In [ ]:
import heytutor

## 1-1. Basics
```
%%hey [problem_file=...] [answer_file=...]
```

### 1-1-1. Builtin variables
* `{file:FILENAME}` is the content of FILE
* `{files:PATTERN}` is the content of files matching PATTERN (e.g., `{files:go/simple/*/*.go}`)
* `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` that of the second last `%%bash_` cell, etc.
* `{problem}` is the content of the file you specified by `%%hey problem_file=foo.md`
* `{answer}` is the content of the file you specified by `%%hey answer_file=go/foo.go`


# 2. C/C++ for comparison

## 2-1. AI tutor

In [ ]:
import heytutor

## 2-2. Set up a module

In [ ]:
%%bash_

mkdir -p cc/simple

## 2-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* If you edit the file outside Jupyter, <font color=red>be careful not to overwrite it with an empty file</font>

In [ ]:
%%writefile_ cc/simple/simple.cc


#include <stdio.h>
#include <stdlib.h>
#include <time.h>

int main(int argc, char ** argv) {
  long s = (1 < argc ? atol(argv[1]) : 1000 * 1000);
  long m = (2 < argc ? atol(argv[2]) : 10);
  long n = (3 < argc ? atol(argv[3]) : m * 10);
  if (sizeof(long) * s * n > (1L << 32)) {
      fprintf(stderr, "you'd better not allocate that much memory\n");
      fprintf(stderr, "sizeof(long)(8) * s(%ld) * n(%ld) < 4GB\n", s, n);
      exit(1);
  }
  volatile long ** a = (volatile long **)malloc(sizeof(long *) * m);
  for (long i = 0; i < n; i++) {
    volatile long * b = (long *)malloc(sizeof(long) * s);
    for (long j = 0; j < s; j++) b[j] = i;
    a[i % m] = b;
  }
  printf("a[0][0] = %ld\n", a[0][0]);
  return 0;
}



In [ ]:
%%bash_

cat cc/simple/simple.cc

## 2-4. Build

In [ ]:
%%bash_

cd cc/simple
g++ -o simple -Wall -Wextra -O3 simple.cc

## 2-5. Run

In [ ]:
%%bash_

lang=cc
exe=cc/simple/simple

S=20000
M=1000
max_n=10000
n=1				# 1 2 3 5 7 10 14 19 ...
while [ $n -le ${max_n} ]; do
    echo "===== lang=${lang} n=${n} ====="
    /usr/bin/time ${exe} ${S} ${M} ${n}
    n=$((n * 4 / 3 + 1))
done


## 2-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=simple.md

Problem:
{problem}

...

# 3. Go

## 3-1. AI tutor

In [ ]:
import heytutor

## 3-2. Set up a module

In [ ]:
%%bash_

export PATH=${PATH}:~/.local/go/bin:~/go/bin
mkdir -p go/simple
cd go/simple
go mod init simple

* Note: when you run `go` or other Go commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/go/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 3-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* If you edit the file outside Jupyter, <font color=red>be careful not to overwrite it with an empty file</font>

In [ ]:
%%writefile_ go/simple/simple.go

package main
import (
    "os"
    "strconv"
    "fmt"
)

func get_int64(args []string, i int, def_val int64) int64 {
	if i < len(args)  {
		x, _ := strconv.Atoi(args[i])
		return int64(x)
	} else {
		return def_val
	}
}

func main() {
	s := get_int64(os.Args, 1, int64(1000 * 1000))
	m := get_int64(os.Args, 2, int64(10))
	n := get_int64(os.Args, 3, m * 10)
	sizeof_int64 := int64(8)
	if sizeof_int64 * s * m > (1 << 30) {
		fmt.Fprintf(os.Stderr, "you'd better not allocate that much memory\n")
		fmt.Fprintf(os.Stderr, "sizeof(int64)(8) * s(%d) * m(%d) = %d > 1GB\n", s, m, sizeof_int64 * s * m)
		os.Exit(1)
	}
	a := make([][]int64, m)
	for i := int64(0); i < n; i++ {
		b := make([]int64, s)
		for j := range b { b[j] = i }
		a[i % m] = b
	}
	fmt.Printf("a[0][0] = %d\n", a[0][0]);
}



In [ ]:
%%bash_

cat go/simple/simple.go

## 3-4. Build

In [ ]:
%%bash_

export PATH=${PATH}:~/.local/go/bin:~/go/bin
cd go/simple
go build

## 3-5. Run

In [ ]:
%%bash_

lang=go
exe=go/simple/simple

S=20000
M=1000
max_n=10000
n=1				# 1 2 3 5 7 10 14 19 ...
while [ $n -le ${max_n} ]; do
    echo "===== lang=${lang} n=${n} ====="
    /usr/bin/time ${exe} ${S} ${M} ${n}
    n=$((n * 4 / 3 + 1))
done


## 3-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=simple.md

Problem:
{problem}

...

# 4. Julia

## 4-1. AI tutor

In [ ]:
import heytutor

## 4-2. Set up a module
* Unnecessary, but make a directory for organization consistent with other languages


In [ ]:
%%bash_

mkdir -p jl/simple

## 4-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* If you edit the file outside Jupyter, <font color=red>be careful not to overwrite it with an empty file</font>

In [ ]:
%%writefile_ jl/simple/simple.jl

#!/usr/bin/env julia
import Printf

function main()
    argc = length(ARGS)
    s = if argc >= 1 parse(Int64, ARGS[1]) else 1000 * 1000 end
    m = if argc >= 2 parse(Int64, ARGS[2]) else 10 end
    n = if argc >= 3 parse(Int64, ARGS[3]) else m * 10 end
    a = Vector{Vector{Int64}}(undef, m)
    for i = 1:n
        b = Vector{Int64}(undef, s)
        fill!(b, i - 1)
        a[(i - 1) % m + 1] = b
    end
    println("a[1][1] = ", a[1][1])
end

main()


    


In [ ]:
%%bash_

cat jl/simple/simple.jl

## 4-4. Build
* Unnecessary, but we make the Julia file executable for consistency with other languages

In [ ]:
%%bash_

chmod +x jl/simple/simple.jl

## 4-5. Run

In [ ]:
%%bash_
export PATH=${PATH}:~/.juliaup/bin

lang=jl
exe=jl/simple/simple.jl

S=20000
M=1000
max_n=10000
n=1				# 1 2 3 5 7 10 14 19 ...
while [ $n -le ${max_n} ]; do
    echo "===== lang=${lang} n=${n} ====="
    /usr/bin/time ${exe} ${S} ${M} ${n}
    n=$((n * 4 / 3 + 1))
done


* Note: when you run `julia` or other Julia commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/.juliaup/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 4-6. Ask Questions or Get Feedback
* Consider including `{bash[-1]}` --- the last output by `%%bash_` --- to get feedback on errors


In [ ]:
%%hey problem_file=simple.md

Problem:
{problem}

...

# 5. OCaml

## 5-1. AI tutor

In [ ]:
import heytutor

## 5-2. Set up a module

In [ ]:
%%bash_

eval $(opam env)
mkdir -p ml
cd ml
dune init proj simple

* Note: when you run `ocamlc` or other OCaml commands (see below) in a terminal (SSH or Jupyter terminal), you need to execute the first line (`eval $(opam env)`)
* You may consider adding that line in your `~/.bash_profile`

## 5-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* `dune init proj simple` command above should have created the following file
* <font color=red>Be careful not to overwrite it with an empty file</font>

In [ ]:
%%bash_

cat ml/simple/bin/main.ml

In [ ]:
%%writefile_ ml/simple/bin/main.ml

let main () = 
  let argv = Sys.argv in
  let argc = Array.length argv in
  let s = if argc > 1 then int_of_string argv.(1) else 1000 * 1000 in
  let m = if argc > 2 then int_of_string argv.(2) else 10 in
  let n = if argc > 3 then int_of_string argv.(3) else m * 10 in
  let a = Array.make m (Array.make 1 0) in
  let rec allocate_loop i =
    if i < n then
      let b = Array.make s i in
      let _ = a.(i mod m) <- b in
      allocate_loop (i + 1)
  in
  allocate_loop 0
;;

main ()



## 5-4. Build

In [ ]:
%%bash_

eval $(opam env)
cd ml/simple
dune build --release

## 5-5. Run

In [ ]:
%%bash_

lang=ml
exe=ml/simple/_build/default/bin/main.exe

S=20000
M=1000
max_n=10000
n=1				# 1 2 3 5 7 10 14 19 ...
while [ $n -le ${max_n} ]; do
    echo "===== lang=${lang} n=${n} ====="
    /usr/bin/time ${exe} ${S} ${M} ${n}
    n=$((n * 4 / 3 + 1))
done


## 5-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=simple.md

Problem:
{problem}

...

# 6. Rust

## 6-1. AI tutor

In [ ]:
import heytutor

## 6-2. Set up a module

In [ ]:
%%bash_

. ~/.cargo/env
mkdir -p rs
cd rs
cargo new simple

* Note: when you run `rustc` or other Rust commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`. ~/.cargo/env`)
* You may consider adding that line in your `~/.bash_profile`

## 6-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* `cargo new simple` command above should have created the following file
* <font color=red>Be careful not to overwrite it with an empty file</font>

In [ ]:
%%bash_

cat rs/simple/src/main.rs

In [ ]:
%%writefile_ rs/simple/src/main.rs

fn main() {
    let args : Vec<String> = std::env::args().collect();
    let argc = args.len();
    let s = if argc > 1 { args[1].parse::<usize>().unwrap() } else { 1_000_000 };
    let m = if argc > 2 { args[2].parse::<usize>().unwrap() } else { 10 };
    let n = if argc > 3 { args[3].parse::<usize>().unwrap() } else { m * 10 };
    let b = Box::new(vec![0; 1]);
    let mut a = Box::new(vec![b; m]);
    for i in 0..n {
        let b = Box::new(vec![i; s]);
        a[i % m] = b;
    }
    println!("a[0][0] = {}", a[0][0]);
}



## 6-4. Build

In [ ]:
%%bash_

. ~/.cargo/env
cd rs/simple
cargo build --release

## 6-5. Run

In [ ]:
%%bash_

lang=rs
exe=rs/simple/target/release/simple

S=20000
M=1000
max_n=10000
n=1				# 1 2 3 5 7 10 14 19 ...
while [ $n -le ${max_n} ]; do
    echo "===== lang=${lang} n=${n} ====="
    /usr/bin/time ${exe} ${S} ${M} ${n}
    n=$((n * 4 / 3 + 1))
done


## 6-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=simple.md

Problem:
{problem}

...


# 7. Summarize your observations in this experiment
* Summarize your observations in this experiment below


In [ ]:
%%writefile_ note.md


# 8. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=simple.md

Problem:
{problem}

My thoughts: note.md
{file:note.md}

Give me feedback on my thoughts.